# nl2cuda-kernel-agent — Colab 驱动入口

在 Colab（运行时选 **T4 GPU**）中依次运行下面的单元格：拉取仓库+装依赖、环境探测、编译冒烟、case 前反向正确性、case 计时。

**架构**：`framework/`（通用，算法无关）+ `cases/<name>/`（每算法一份）。RBF 是第一个 case。

**协作方式**：运行后把输出贴回给助手，助手改代码 push，你再 pull 重跑。

## 1. 拉取 / 更新仓库 + 安装依赖
首次 clone；之后 pull。并装 `ninja`（CUDA 扩展即时编译所需，Colab 默认未装）。

In [ ]:
import os

REPO_URL = 'https://github.com/SilenceWanna/nl2cuda-kernel-agent.git'
REPO_DIR = '/content/nl2cuda-kernel-agent'

if os.path.isdir(REPO_DIR):
    print('仓库已存在，执行 git pull ...')
    !cd {REPO_DIR} && git pull --ff-only
else:
    print('clone 仓库 ...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
!git log --oneline -3

print('\n安装 ninja（CUDA 扩展即时编译所需）...')
!pip install -q ninja
import shutil
print('ninja 可用:', shutil.which('ninja') is not None)

## 2. 环境探测
确认 GPU 型号、CUDA / PyTorch 版本、TF32 支持等。

In [ ]:
!python scripts/probe_env.py

## 3. 编译链路冒烟测试
编译最小 `framework/smoke.cu`（向量加法）并验证，确认 nvcc + cpp_extension + sm_75 编译链路可用。

In [ ]:
!python framework/smoke_test.py

## 4. Case 前反向正确性
编译 `cases/rbf/kernels/` 的前反向 kernel，对拍参考实现的**前向与反向**（≥5 种子 allclose）。应全 **PASS**。
（kernel float32；反向正确性以对拍参考 autograd 的 allclose 为准，与验收判据一致。）

> 换其他算法只需改 `--case <name>`。

In [ ]:
!python skill/scripts/verify_case.py --case rbf

## 5. Case 计时（vs torch.compile）
候选 kernel vs `torch.compile(参考)` 前反向计时。朴素版此时打不过（优化留阶段3），用于摸差距。
baseline 参考（RBF 2048², T4）：前向 ~1.55ms、反向 ~8ms。

> 首次含 `torch.compile` 编译，稍慢。

In [ ]:
!python skill/scripts/bench_case.py --case rbf